# Organizing budget data
These codes in this file aim to organize budget data.   
That's because processed data is recording all operations (e.g. add, repeal and amend) of trailer bills and has some duplication.

In [1]:
import pandas as pd

In [3]:
# Designate a target csv file
processed_data = "2023_budget_bill_final.csv"

In [4]:
# Convert csv files into dataframes
target_df = pd.read_csv(processed_data)
target_df.head()

,level,agency,item_number,program_code,object_code,fund_code,fund_name,schedule_group,name,amount_numeric,revised_amount,trailer_bill_source,action_type
0,program,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,960.0,EMPTY,1.0,NaN,1.0,Support of the Senate,177325000.0,177325000,Original,Original
1,object,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,960.0,101001.0,1.0,NaN,1.0,Salaries of Senators,-6751000.0,-6751000,Original,Original
2,object,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,960.0,317295.0,1.0,NaN,1.0,Mileage,-11000.0,-11000,Original,Original
3,object,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,960.0,317292.0,1.0,NaN,1.0,Expenses,-1712000.0,-1712000,Original,Original
4,object,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,960.0,500004.0,1.0,NaN,1.0,Operating Expenses,-168851000.0,-168851000,Original,Original


In [9]:
# ---------------------------------------------------------
# Define the unique keys for budget line items
# ---------------------------------------------------------
unique_keys = ['item_number', 'program_code', 'object_code', 'fund_code']

In [12]:
# ---------------------------------------------------------
# Step 9-1: Aggregate amounts within the Latest Source
# We group by these keys to sum Base Amounts and Reimbursements
# ---------------------------------------------------------
df_summarized = target_df.groupby(unique_keys, as_index=False).agg({
    'revised_amount': 'sum',
    'amount_numeric': 'sum', 
    'agency': 'first',
    'fund_name': 'first',
    'trailer_bill_source': 'first'
})

# ---------------------------------------------------------
# Step 9-2: Identify items that have detailed schedules
# ---------------------------------------------------------
items_with_details = set(
    df_summarized[
        (df_summarized["program_code"] != "EMPTY") | 
        (df_summarized["object_code"] != "EMPTY")
    ]["item_number"]
)

# ---------------------------------------------------------
# Step 9-3: Apply the logic to filter out Subtotals (Point ③)
# ---------------------------------------------------------
def filter_subtotals(row):
    # If a row is a subtotal (both codes are EMPTY)
    if row["program_code"] == "EMPTY" and row["object_code"] == "EMPTY":
        # Only keep it if NO other detailed rows exist for this Item
        return row["item_number"] not in items_with_details
    return True

df_final_snapshot = df_summarized[df_summarized.apply(filter_subtotals, axis=1)].copy()

# ---------------------------------------------------------
# Step 9-4: Export the final results to CSV
# ---------------------------------------------------------
df_final_snapshot.to_csv("2023_Budget_Final_Calculated.csv", index=False, encoding='utf-8-sig')

print(f"Final records exported: {len(df_final_snapshot)}")

Final records exported: 2738
